In [0]:
%run /Workspace/Users/antoniorad15@gmail.com/ROBOTICS-AI-training-pipeline/pipeline-finetune-gr00t/secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_2", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_2", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [0]:
from datetime import datetime
import os

RUN_ID = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
os.environ["RUN_ID"] = RUN_ID  # makes it available to %sh cells
PIPELINE_START = datetime.utcnow()
print(f"RUN_ID: {RUN_ID}")

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
python3 --version
pip3 install h5py opencv-python-headless ffmpeg-python static-ffmpeg imageio pyarrow
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo rm -f /usr/local/bin/ffmpeg /usr/local/bin/ffprobe
sudo apt-get update -qq && sudo apt-get install -y ffmpeg
echo "=== ffmpeg version ==="
ffmpeg -version | head -1
echo "=== libx264 support ==="
ffmpeg -encoders 2>/dev/null | grep libx264 || echo "WARNING: libx264 NOT available"
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
mkdir -p /ephemeral/hdf5_datasets /ephemeral/lerobot_datasets /ephemeral/scripts /ephemeral/conversion_metrics
mkdir -p /ephemeral/scripts/logs
cd scripts
mkdir logs 
EOF

In [0]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf5_test/Top_Long_Seen_0-generated-HALTON_3_KEYPOINTS_PINOCCHIO.hdf5 \
  $SSH_USER@$BREV_IP:/ephemeral/hdf5_datasets/

In [ ]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf52lerobot_script_files_metrics/ephemeral_convert_videos_hdf5_lerobot_many_clothes_matrix.py \
  $SSH_USER@$BREV_IP:/ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP \
  "sed -i \"s/RUN_ID = \\\"\\\"/RUN_ID = \\\"$RUN_ID\\\"/\" /ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py"

# Patch: GPU encoder -> CPU libx264, ProcessPoolExecutor -> ThreadPoolExecutor, suppress stdout
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'SSHEOF'
cat > /tmp/fix_script.py << 'PYEOF'
content = open('/ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py').read()
content = content.replace('from concurrent.futures import ProcessPoolExecutor, as_completed', 'from concurrent.futures import ThreadPoolExecutor as ProcessPoolExecutor, as_completed')
content = content.replace('stdin=subprocess.PIPE, stderr=subprocess.DEVNULL', 'stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL')
content = content.replace('h264_nvenc', 'libx264')
open('/ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py', 'w').write(content)
print('Done')
PYEOF
python3 /tmp/fix_script.py
echo "=== Verify ==="
grep -n 'ThreadPoolExecutor\|stdout=subprocess\|c:v' /ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py
SSHEOF

In [0]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf52lerobot_script_files_metrics/ephemeral_convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.py \
  $SSH_USER@$BREV_IP:/ephemeral/scripts/convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.py

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP \
  "sed -i 's/TASK_NAME = \"your_task_name\"/TASK_NAME = \"$TASK_NAME\"/' /ephemeral/scripts/convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.py"

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP \
  "sed -i \"s/RUN_ID = \\\"\\\"/RUN_ID = \\\"$RUN_ID\\\"/\" /ephemeral/scripts/convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.py"

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
tmux kill-session -t work 2>/dev/null
tmux new-session -d -s work
tmux send-keys -t work "python3 /ephemeral/scripts/convert_videos_hdf5_lerobot_many_clothes_matrix.py 2>&1 | tee /ephemeral/scripts/logs/convert_videos_hdf5_lerobot_many_clothes_matrix.log && python3 /ephemeral/scripts/convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.py 2>&1 | tee /ephemeral/scripts/logs/convert_data_and_meta_hdf5_lerobot_many_clothes_matrix.log" Enter
tmux send-keys -t work "exit" Enter
EOF

In [0]:
%sh
echo "Waiting for data conversion to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t work 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Data conversion complete!"
    break
  fi
  
  sleep 60
done

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/lerobot_datasets/
du -sh .
ls -lh
EOF

In [ ]:
import subprocess, json, os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import time 
pipeline_end_ts = datetime.utcnow().isoformat()

BREV_IP  = dbutils.secrets.get("brev", "brev_ip")
SSH_USER = os.environ["SSH_USER"]
KEY_PATH = "/tmp/ssh_private_key_2"

# SCP both metric files from Brev
for fname in [f"video_metrics_{RUN_ID}.json", f"data_metrics_{RUN_ID}.json"]:
    subprocess.run([
        "scp", "-i", KEY_PATH, "-o", "StrictHostKeyChecking=no",
        f"{SSH_USER}@{BREV_IP}:~/conversion_metrics/{fname}",
        f"/tmp/{fname}"
    ], check=True)

# --- Video metrics schema ---
video_schema = StructType([
    StructField("run_id",           StringType(),  True),
    StructField("hdf5_source",      StringType(),  True),
    StructField("episode_id",       IntegerType(), True),
    StructField("camera",           StringType(),  True),
    StructField("num_frames",       IntegerType(), True),
    StructField("resolution",       StringType(),  True),
    StructField("encode_time_sec",  DoubleType(),  True),
    StructField("encode_fps",       DoubleType(),  True),
    StructField("video_size_mb",    DoubleType(),  True),
    StructField("status",           StringType(),  True),
    StructField("error_msg",        StringType(),  True),
    StructField("ts",               StringType(),  True),
])

with open(f"/tmp/video_metrics_{RUN_ID}.json") as f:
    video_rows = json.load(f)

spark.createDataFrame(video_rows, schema=video_schema) \
    .write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.video_conversion_metrics")

# --- Data metrics schema ---
from pyspark.sql.types import ArrayType, FloatType

data_schema = StructType([
    StructField("run_id",                   StringType(),              True),
    StructField("hdf5_source",              StringType(),              True),
    StructField("episode_id",               IntegerType(),             True),
    StructField("num_frames",               IntegerType(),             True),
    StructField("duration_sec",             DoubleType(),              True),
    StructField("parquet_size_mb",          DoubleType(),              True),
    StructField("parquet_write_time_sec",   DoubleType(),              True),
    StructField("action_mean",              ArrayType(DoubleType()),   True),
    StructField("action_std",               ArrayType(DoubleType()),   True),
    StructField("obs_mean",                 ArrayType(DoubleType()),   True),
    StructField("obs_std",                  ArrayType(DoubleType()),   True),
    StructField("ts",                       StringType(),              True),
])

with open(f"/tmp/data_metrics_{RUN_ID}.json") as f:
    payload = json.load(f)

spark.createDataFrame(payload["episodes"], schema=data_schema) \
    .write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.data_conversion_metrics")

total_duration_sec = round((datetime.utcnow() - PIPELINE_START).total_seconds(), 2)
payload["summary"]["total_duration_sec"] = total_duration_sec
print(f"Total pipeline duration: {total_duration_sec}s ({total_duration_sec/60:.1f} min)")

# --- Pipeline runs schema ---
summary_schema = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("hdf5_size_mb",             DoubleType(),  True),
    StructField("total_episodes",           IntegerType(), True),
    StructField("total_frames",             IntegerType(), True),
    StructField("total_duration_sec",       DoubleType(),  True),
    StructField("avg_frames_per_episode",   DoubleType(),  True),
    StructField("ts",                       StringType(),  True),
])

spark.createDataFrame([payload["summary"]], schema=summary_schema) \
    .write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.pipeline_runs")

print(f"Metrics for run {RUN_ID} loaded into Delta.")

In [0]:
%sh
rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no" \
  $SSH_USER@$BREV_IP:~/lerobot_datasets/ \
  /Volumes/workspace/default/lerobot_datasets_lehome_many_clothes/